# Phase 5.1 — Final Training

Freeze the practical hybrid recommendation pipeline selected in Phase 3.4 and save the model components required for serving.

Final model strategy:
- Collaborative latent factors
- Content metadata representation
- Popularity fallback
- Recency signal
- Fixed hybrid scoring configuration

Artifacts are written under `artifacts/` and should remain outside Git if they are large.


In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

TRAIN_UI_PATH = PROCESSED_DIR / "train_user_item.csv"
TRAIN_PATH = PROCESSED_DIR / "train_interactions.csv"

print("Project root:", PROJECT_ROOT)
print("Artifacts:", ARTIFACTS_DIR)


Project root: f:\annuspeaks.com\recommendation-system
Artifacts: f:\annuspeaks.com\recommendation-system\artifacts


## 5.1.1 Freeze Final Feature / Preprocessing Pipeline

The final pipeline uses the already-established training representation:

`user_id → item_id → total behavioral weight`

and the product metadata representation:

`property=value` tokens → TF-IDF.

No raw files are modified.


In [2]:
train_ui = pd.read_csv(
    TRAIN_UI_PATH,
    usecols=["user_id", "item_id", "total_weight", "last_timestamp"],
)

print("Training user-item rows:", f"{len(train_ui):,}")
print("Users:", f"{train_ui['user_id'].nunique():,}")
print("Products:", f"{train_ui['item_id'].nunique():,}")

# Freeze the preprocessing/configuration contract.
pipeline_config = {
    "feedback": "implicit",
    "interaction_weight_column": "total_weight",
    "timestamp_column": "last_timestamp",
    "top_k": 10,
    "svd_components": 32,
    "hybrid_weights": {
        "collaborative": 0.45,
        "content": 0.30,
        "popularity": 0.15,
        "recency": 0.10,
    },
    "cold_start_user": "popularity",
    "cold_start_product": "content_metadata",
}

with open(ARTIFACTS_DIR / "pipeline_config.json", "w", encoding="utf-8") as f:
    json.dump(pipeline_config, f, indent=2)

print("Pipeline configuration frozen.")


Training user-item rows: 1,939,777
Users: 1,407,580
Products: 228,392
Pipeline configuration frozen.


## 5.1.2 Train Final Collaborative Component

Train the matrix-factorization component on the complete available training representation.

The factor count remains 32 to keep the final artifact practical for the later API layer.


In [3]:
# Build the final sparse user-item matrix.

user_ids = train_ui["user_id"].unique()
item_ids = train_ui["item_id"].unique()

user_to_index = {
    user_id: idx for idx, user_id in enumerate(user_ids)
}

item_to_index = {
    item_id: idx for idx, item_id in enumerate(item_ids)
}

rows = train_ui["user_id"].map(user_to_index).to_numpy()
cols = train_ui["item_id"].map(item_to_index).to_numpy()
values = train_ui["total_weight"].to_numpy(dtype=np.float32)

user_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(item_ids)),
    dtype=np.float32,
)

n_components = min(
    32,
    min(user_item_matrix.shape) - 1,
)

svd = TruncatedSVD(
    n_components=n_components,
    algorithm="randomized",
    n_iter=3,
    random_state=42,
)

user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T

print("Final matrix:", user_item_matrix.shape)
print("Final latent dimensions:", n_components)
print(
    "Explained variance:",
    f"{svd.explained_variance_ratio_.sum():.4f}"
)


Final matrix: (1407580, 228392)
Final latent dimensions: 32
Explained variance: 0.1451


## 5.1.3 Save Final Collaborative Artifacts

Save the mappings and latent factors required to score personalized recommendations.


In [4]:
np.save(ARTIFACTS_DIR / "user_ids.npy", user_ids)
np.save(ARTIFACTS_DIR / "item_ids.npy", item_ids)
np.save(ARTIFACTS_DIR / "user_factors.npy", user_factors.astype(np.float32))
np.save(ARTIFACTS_DIR / "item_factors.npy", item_factors.astype(np.float32))

with open(ARTIFACTS_DIR / "user_to_index.pkl", "wb") as f:
    pickle.dump(user_to_index, f)

with open(ARTIFACTS_DIR / "item_to_index.pkl", "wb") as f:
    pickle.dump(item_to_index, f)

# Save the sparse training representation as an optional reproducibility artifact.
save_npz(
    ARTIFACTS_DIR / "user_item_matrix.npz",
    user_item_matrix,
)

print("Collaborative artifacts saved.")


Collaborative artifacts saved.


## 5.1.4 Freeze Popularity and Recency Components

Save the catalog-level fallback and recency signals used by the hybrid scorer.


In [5]:
popularity = (
    train_ui
    .groupby("item_id")["total_weight"]
    .sum()
    .sort_values(ascending=False)
)

item_recency = (
    train_ui
    .groupby("item_id")["last_timestamp"]
    .max()
)

popularity.to_csv(
    ARTIFACTS_DIR / "item_popularity.csv",
    header=["popularity_score"],
)

item_recency.to_csv(
    ARTIFACTS_DIR / "item_recency.csv",
    header=["last_timestamp"],
)

print("Popularity and recency artifacts saved.")


Popularity and recency artifacts saved.


## 5.1.5 Freeze Content Preprocessing

Build and save the product metadata TF-IDF preprocessing used by the content component.

The sparse TF-IDF matrix is saved separately so it can be loaded/rebuilt by the serving layer without retraining the vectorizer.


In [6]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"

properties_parts = []

for path in [
    RAW_DIR / "item_properties_part1.csv",
    RAW_DIR / "item_properties_part2.csv",
]:
    for chunk in pd.read_csv(
        path,
        usecols=["itemid", "property", "value"],
        chunksize=250_000,
    ):
        chunk = chunk.dropna(
            subset=["itemid", "property", "value"]
        ).copy()

        chunk["property"] = chunk["property"].astype(str).str.strip()
        chunk["value"] = chunk["value"].astype(str).str.strip()

        chunk["token"] = (
            chunk["property"]
            + "="
            + chunk["value"]
        )

        properties_parts.append(
            chunk[["itemid", "token"]]
        )

properties = pd.concat(
    properties_parts,
    ignore_index=True
).drop_duplicates()

product_text = (
    properties
    .groupby("itemid")["token"]
    .agg(" ".join)
    .reset_index()
    .rename(columns={"itemid": "item_id"})
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+=\S+\b",
    min_df=2,
    max_features=50_000,
)

tfidf_matrix = vectorizer.fit_transform(
    product_text["token"]
)

print("Content products:", f"{len(product_text):,}")
print("TF-IDF shape:", tfidf_matrix.shape)


Content products: 417,053
TF-IDF shape: (417053, 50000)


In [7]:
# Save content artifacts.

np.save(
    ARTIFACTS_DIR / "content_item_ids.npy",
    product_text["item_id"].to_numpy(),
)

save_npz(
    ARTIFACTS_DIR / "product_tfidf.npz",
    tfidf_matrix,
)

with open(
    ARTIFACTS_DIR / "tfidf_vectorizer.pkl",
    "wb",
) as f:
    pickle.dump(vectorizer, f)

print("Content preprocessing artifacts saved.")


Content preprocessing artifacts saved.


## 5.1.6 Save Final Model Manifest

Create one manifest describing the final artifact set and the serving configuration.


In [8]:
artifact_files = sorted(
    path.name
    for path in ARTIFACTS_DIR.iterdir()
    if path.is_file()
)

manifest = {
    "model": "hybrid_recommendation_engine",
    "version": "1.0",
    "components": [
        "collaborative_matrix_factorization",
        "content_tfidf",
        "popularity",
        "recency",
    ],
    "artifact_directory": "artifacts/",
    "artifact_files": artifact_files,
    "top_k": 10,
    "hybrid_weights": pipeline_config["hybrid_weights"],
}

with open(
    ARTIFACTS_DIR / "model_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest, f, indent=2)

print("Model manifest saved.")
print("Artifacts:", len(artifact_files))


Model manifest saved.
Artifacts: 13


## 5.1.7 Final Artifact Validation

Verify that the saved model components can be loaded and that the configuration is internally consistent.


In [9]:
# Reload critical artifacts.

loaded_user_factors = np.load(
    ARTIFACTS_DIR / "user_factors.npy"
)

loaded_item_factors = np.load(
    ARTIFACTS_DIR / "item_factors.npy"
)

loaded_user_ids = np.load(
    ARTIFACTS_DIR / "user_ids.npy"
)

loaded_item_ids = np.load(
    ARTIFACTS_DIR / "item_ids.npy"
)

with open(
    ARTIFACTS_DIR / "pipeline_config.json",
    encoding="utf-8",
) as f:
    loaded_config = json.load(f)

with open(
    ARTIFACTS_DIR / "model_manifest.json",
    encoding="utf-8",
) as f:
    loaded_manifest = json.load(f)

assert loaded_user_factors.shape == (
    len(loaded_user_ids),
    n_components,
)

assert loaded_item_factors.shape == (
    len(loaded_item_ids),
    n_components,
)

assert loaded_config["hybrid_weights"] == {
    "collaborative": 0.45,
    "content": 0.30,
    "popularity": 0.15,
    "recency": 0.10,
}

assert loaded_manifest["model"] == "hybrid_recommendation_engine"
assert len(artifact_files) >= 8

print("Phase 5.1 validation: PASS")
print("Final model:", loaded_manifest["model"])
print("User factors:", loaded_user_factors.shape)
print("Item factors:", loaded_item_factors.shape)
print("Artifact count:", len(artifact_files))


Phase 5.1 validation: PASS
Final model: hybrid_recommendation_engine
User factors: (1407580, 32)
Item factors: (228392, 32)
Artifact count: 13


## Phase 5.1 Completion

- Final preprocessing/configuration frozen.
- Final collaborative component trained on the complete training representation.
- Popularity and recency signals saved.
- Content TF-IDF preprocessing and matrix saved.
- Required mappings, embeddings/factors, and configuration saved.
- Final model manifest created and validated.
